# Multi-Input Operations

## What you'll learn

- Why operations sometimes need multiple input roles
- How `group_by` strategies control which artifacts get paired together
- How LINEAGE pairs a derived artifact with its nearest matching ancestor
- How LINEAGE handles 1:N expansion for parameter sweeps
- How to override the pairing strategy per step at `pipeline.run()`

**Prerequisites:** [Sources and Sequencing](01-sources-and-sequencing.ipynb), [Branching and Merging](02-branching-and-merging.ipynb)
**Estimated time:** 15 minutes

---

An operation may need a dataset and a config file, or outputs from two branches.
Its `group_by` strategy determines which artifacts from these input roles belong
together. Here `LINEAGE` pairs each config with the dataset it came from.

In [ ]:
from __future__ import annotations

from artisan.operations.examples import (
    DataGenerator,
    DataTransformerConfig,
    DataTransformerScript,
)
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import build_macro_graph, build_micro_graph

> **Graph legend:** See [Sources and Sequencing](01-sources-and-sequencing.ipynb) for box/arrow key.

## Choose which inputs belong together

Here each config is derived from a dataset, so `LINEAGE` uses that relationship
to pair them. If independent streams share a filename convention instead, use
[Name-Based Pairing](05-name-based-pairing.ipynb).

An operation can declare its pairing default. A pipeline step can override it
with `group_by`, as shown in [Step Overrides](../05-errors-and-control/01-step-overrides.ipynb).
For an all-combinations parameter sweep, that tutorial uses `CROSS_PRODUCT`.

## LINEAGE pairing

LINEAGE walks backward from each candidate to its nearest target ancestor.
Two artifacts that only share a common ancestor, such as sibling branch
outputs, do not match.

In this config-and-execute example:

- `DataGenerator` produces 3 datasets
- `DataTransformerConfig` reads each dataset and produces a config for it
- `DataTransformerScript` needs both the dataset *and* its config

Each config descends from a specific dataset, so the framework pairs
dataset A with config A, dataset B with config B, and so on.

In [ ]:
env_lineage = tutorial_setup("lineage_basic")


def run_lineage_basic():
    pipeline = PipelineManager.create(
        name="lineage_basic",
        delta_root=env_lineage.delta_root,
        staging_root=env_lineage.staging_root,
        working_root=env_lineage.working_root,
    )
    output = pipeline.output

    # Step 0: Generate 3 datasets
    pipeline.run(
        operation=DataGenerator, name="generate", params={"count": 3, "seed": 42}
    )

    # Step 1: Produce a config for each dataset (1:1 lineage)
    pipeline.run(
        operation=DataTransformerConfig,
        name="configure",
        inputs={"dataset": output("generate", "datasets")},
        params={"scale_factors": [0.5]},
    )

    # Step 2: Execute with paired dataset + config (multi-input, group_by=LINEAGE)
    pipeline.run(
        operation=DataTransformerScript,
        name="execute",
        inputs={
            "dataset": output("generate", "datasets"),
            "config": output("configure", "config"),
        },
    )

    return pipeline.finalize()


result_lineage = run_lineage_basic()

The macro graph shows two inputs to step 2 (`execute`): datasets from step 0
and configs from step 1.

In [ ]:
build_macro_graph(env_lineage.delta_root)

The orange lineage arrows connect each config to its source dataset. LINEAGE
pairing follows those relationships. To inspect an earlier stage, rerun the
graph cell with `max_step=1`; `max_step=2` includes the paired step.

In [ ]:
build_micro_graph(env_lineage.delta_root)

## LINEAGE with 1:N expansion (parameter sweep)

`DataTransformerConfig` creates one config per combination of `scale_factors`
and `noise_amplitudes`. LINEAGE pairs each config with its source dataset.

With 1 dataset, 3 scale factors, and 2 noise amplitudes, the step produces
6 execution units. Each pairs the dataset with one config.

In [ ]:
env_sweep = tutorial_setup("lineage_sweep")


def run_lineage_sweep():
    pipeline = PipelineManager.create(
        name="lineage_sweep",
        delta_root=env_sweep.delta_root,
        staging_root=env_sweep.staging_root,
        working_root=env_sweep.working_root,
    )
    output = pipeline.output

    # 1 dataset
    pipeline.run(
        operation=DataGenerator, name="generate", params={"count": 1, "seed": 42}
    )

    # 6 configs (3 scale_factors × 2 noise_amplitudes)
    pipeline.run(
        operation=DataTransformerConfig,
        name="configure",
        inputs={"dataset": output("generate", "datasets")},
        params={
            "scale_factors": [0.1, 0.5, 1.0],
            "noise_amplitudes": [1, 2],
        },
    )

    # 6 execution units (each dataset paired with one config via lineage)
    pipeline.run(
        operation=DataTransformerScript,
        name="execute",
        inputs={
            "dataset": output("generate", "datasets"),
            "config": output("configure", "config"),
        },
    )

    return pipeline.finalize()


result_sweep = run_lineage_sweep()

In [ ]:
build_macro_graph(env_sweep.delta_root)

The graph shows six configs alongside one dataset. Each config traces back to
that same source dataset. Set `max_step=1` in the graph call to inspect the
configs before the paired operation runs.

In [ ]:
build_micro_graph(env_sweep.delta_root)

### Scaling to multiple datasets

With 3 datasets and 2 scale factors, `DataTransformerConfig` produces 6 configs.
LINEAGE pairs each dataset with its 2 configs, yielding 6 execution units.
A cross product would yield 18.

In [ ]:
env_multi = tutorial_setup("lineage_multi_dataset")


def run_lineage_multi_dataset():
    pipeline = PipelineManager.create(
        name="lineage_multi_dataset",
        delta_root=env_multi.delta_root,
        staging_root=env_multi.staging_root,
        working_root=env_multi.working_root,
    )
    output = pipeline.output

    # 3 datasets
    pipeline.run(
        operation=DataGenerator, name="generate", params={"count": 3, "seed": 42}
    )

    # 2 configs per dataset = 6 configs total
    pipeline.run(
        operation=DataTransformerConfig,
        name="configure",
        inputs={"dataset": output("generate", "datasets")},
        params={"scale_factors": [0.5, 2.0]},
    )

    # 6 execution units (each dataset paired with its 2 configs)
    pipeline.run(
        operation=DataTransformerScript,
        name="execute",
        inputs={
            "dataset": output("generate", "datasets"),
            "config": output("configure", "config"),
        },
    )

    return pipeline.finalize()


result_multi = run_lineage_multi_dataset()

In [ ]:
build_macro_graph(env_multi.delta_root)

In [ ]:
build_micro_graph(env_multi.delta_root)

Each dataset pairs only with its own derived configs: dataset 0 with configs 0
and 1, dataset 1 with configs 2 and 3, and so on. Inspect these arrows in the
graph, or add `max_step=1` to view the state before pairing.
[Provenance Graphs](../08-analysis/01-provenance-graphs.ipynb) shows these snapshots
in more detail.

This example supplies the dataset and config as separate roles and pairs them
by declared ancestry. An operation can instead opt into typed config
references with `InputSpec.resolve_references` and read the prepared
dependencies through `inputs.referenced_artifacts(config)`. Reference
resolution does not declare output parents or change input pairing. See
[Writing Creator Operations](../../how-to-guides/writing-creator-operations.md)
for the reference-based input contract.

## Next steps

- [Name-Based Pairing](05-name-based-pairing.ipynb) — Pair independent streams by filename stem
- [Step Overrides](../05-errors-and-control/01-step-overrides.ipynb) — Choose a different pairing strategy for a step
- [Operations Model](../../concepts/operations-model.md) — Understand input pairing
- [Writing Creator Operations](../../how-to-guides/writing-creator-operations.md) — Declare inputs in an operation